# 08 · Front-end impairments

Between the DSP and the channel sit analog components: the IQ modulator,
DACs, drivers, the 90° hybrid, photodiodes and ADCs.  Their imperfections
are deterministic and mostly simple.  This notebook covers the classic
ones, each with how a receiver removes it:

1. **IQ imbalance**: unequal gains and a non-90° angle between I and Q;
2. **DC offsets**;
3. **limited electrical bandwidth**, which causes ISI;
4. **carrier frequency offset** between the transmitter laser and the LO.

The chain where they all appear together is
[19 · End-to-end coherent transceiver](19-coherent-transceiver-end-to-end.ipynb).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RRC, Constellation
from commkit.backend import to_device
from commkit.impairments import GramSchmidt, Lowdin
from commkit.recovery import DataAided

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
C16 = Constellation.qam(16)


def host(a):
    return to_device(a, "cpu")


def snr_db(y):
    """SNR after removing a static rotation against the known symbols.

    Metrics never rotate on their own; DataAided() fits one common phase.
    """
    return ck.metrics.snr(ck.recovery.correct_carrier_phase(y, DataAided()))

## 1. IQ imbalance

An ideal receiver measures `I = Re{r}` and `Q = Im{r}` with equal gains on
orthogonal axes.  A real one has an amplitude imbalance `ε` (in dB) and a
phase error `φ`, which mixes in the **mirror image** of the signal:

$$r = \mu\, s + \nu\, s^*,$$

with an image-rejection ratio `|μ|²/|ν|²` of about 25-30 dB for typical
devices (the example below exaggerates it).  The square grid of a QAM constellation becomes a skewed
parallelogram.

In [ ]:
tx = ck.generate(C16, num_symbols=2**15, symbol_rate=32e9, rng=1, device=DEVICE)
rx = ck.impairments.apply_awgn(tx, esn0_db=25, rng=2)
iq = ck.impairments.apply_iq_imbalance(
    rx, amplitude_imbalance_db=1.5, phase_imbalance_deg=8.0
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
ck.plotting.plot_constellation(rx, ax=axes[0], title="Balanced receiver")
ck.plotting.plot_constellation(iq, ax=axes[1], title="1.5 dB, 8° IQ imbalance")
plt.show()

# The image term: a single tone at +f acquires a mirror at -f.
fs = 1e6
t = np.arange(2**14) / fs
tone = ck.backend.to_device(np.exp(2j * np.pi * 100e3 * t).astype(np.complex64), DEVICE)
tone_iq = ck.impairments.apply_iq_imbalance(
    tone, amplitude_imbalance_db=1.5, phase_imbalance_deg=8.0
)
spec = np.abs(np.fft.fft(host(tone_iq))) ** 2
k = int(round(100e3 / fs * t.size))
print(f"image rejection ratio: {10 * np.log10(spec[k] / spec[-k]):.1f} dB")

### Blind correction

Because a QAM (or any circular) signal has I and Q of equal power and
uncorrelated, the imbalance can be removed blindly by re-orthogonalizing
the two branches:

* `GramSchmidt()` keeps I as the reference, removes its projection from Q
  and normalizes both (the classic GSOP);
* `Lowdin()` treats both branches symmetrically (whitening of the 2x2
  covariance), spreading the correction evenly.

Both restore the SNR.  They differ by a small common rotation (GSOP keeps
the I axis, Löwdin splits the difference), which carrier recovery removes
anyway; the SNRs below are measured after a data-aided derotation.

In [ ]:
print(f"SNR balanced         {snr_db(rx):5.2f} dB")
print(f"SNR with imbalance   {snr_db(iq):5.2f} dB")
for how in (GramSchmidt(), Lowdin()):
    fixed = ck.impairments.correct_iq_imbalance(iq, how)
    print(f"SNR after {type(how).__name__:12s} {snr_db(fixed):5.2f} dB")

## 2. DC offsets

Imperfectly balanced photodiode pairs and ADC offsets add a constant to I
and Q: a spectral line at 0 Hz and a shifted constellation.  It is removed
by subtracting the mean, *before* the IQ-imbalance correction (which assumes
a zero-mean signal).  In self-homodyne measurements the line sits in the
middle of the signal and needs a dark capture instead (see
[23](23-laser-linewidth-self-homodyne-iq.ipynb)).

In [ ]:
dc = 0.15 - 0.1j
off = iq.replace(samples=iq.samples + dc)
restored = ck.impairments.correct_iq_imbalance(
    off.replace(samples=off.samples - off.samples.mean()), GramSchmidt()
)
print(f"estimated DC {complex(off.samples.mean()):.3f} (true {dc})")
print(
    f"SNR: with DC and imbalance {snr_db(off):.2f} dB, corrected {snr_db(restored):.2f} dB"
)

## 3. Limited electrical bandwidth

DACs, drivers and photodiodes low-pass filter the waveform.  A bandwidth
approaching half the symbol rate widens each pulse into its neighbours,
and the eye closes: ISI.  Modeled here as a 4th-order Bessel response
(03 · section 3) with its -3 dB point at `cutoff` (`norm="mag"`), on a
32 GBd RRC signal at 2 samples per symbol.  The adaptive
equalizer removes this ISI later
([14 · Adaptive equalization](14-adaptive-equalization.ipynb)).

In [ ]:
tx2 = ck.generate(
    C16,
    num_symbols=2**15,
    symbol_rate=32e9,
    sps=2,
    pulse=RRC(rolloff=0.1),
    rng=3,
    device=DEVICE,
)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, bw in zip(axes, (None, 20e9, 12e9)):
    x = (
        tx2
        if bw is None
        else ck.filtering.iir_filter(
            tx2,
            ck.filtering.bessel_sos(
                sampling_rate=tx2.sampling_rate, cutoff=bw, order=4, norm="mag"
            ),
        )
    )
    y = ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(x))
    label = "no bandwidth limit" if bw is None else f"Bessel {bw / 1e9:.0f} GHz"
    print(f"{label:20s} signal-to-ISI {ck.metrics.snr(y):5.1f} dB")
    ck.plotting.plot_constellation(y, ax=ax, title=label)
plt.show()

## 4. Carrier frequency offset

The transmitter laser and the LO are independent, and their frequencies
differ by up to a few GHz.  The received baseband is shifted by that
offset `Δf`: the constellation **spins** at `Δf` turns per second, which
over one symbol is `2π Δf T` radians.  Even 100 MHz at 32 GBd is a full
turn every 320 symbols, far faster than phase noise; frequency-offset
estimation removes it first
([10 · Frequency-offset estimation](10-frequency-offset-estimation.ipynb)).

In [ ]:
spun = ck.spectral.shift_frequency(tx, frequency=100e6)
phase = np.unwrap(np.angle(host(spun.samples) * np.conj(host(tx.samples))))
slope = np.polyfit(np.arange(phase.size), phase, 1)[0]
print(
    f"phase advance per symbol {slope:.5f} rad, 2π Δf T = {2 * np.pi * 100e6 / 32e9:.5f} rad"
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
ck.plotting.plot_constellation(
    spun.samples[:100], ax=axes[0], title="First 100 symbols", constellation=C16
)
ck.plotting.plot_constellation(spun, ax=axes[1], title="Whole record: a ring")
plt.show()

**Summary**

| impairment | signature | removed by |
|---|---|---|
| IQ imbalance | image at `-f`, skewed grid | `correct_iq_imbalance(GramSchmidt() or Lowdin())` |
| DC offset | line at 0 Hz, shifted grid | mean subtraction (or a dark capture) |
| bandwidth limit | ISI, closed eye | adaptive equalizer |
| frequency offset | spinning constellation | `correct_frequency_offset` |

**Next:** [09 · Frame synchronization and timing](09-frame-synchronization-and-timing.ipynb).